## Librerías.

In [ ]:
import sys
import os
import requests
import zipfile
import shutil
import glob
import pandas as pd
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
project_path = '/content/drive/MyDrive/Redes neuronales/Tarea 7. Reconocimiento facial'

if project_path not in sys.path:
    sys.path.append(project_path)
    print(f"Ruta añadida al sistema: {project_path}")

Ruta añadida al sistema: /content/drive/MyDrive/Redes neuronales/Tarea 7. Reconocimiento facial


In [ ]:
# Traemos el código del modelo.
from src.models import construir_cnn_base

## Descarga manual y procesamiento de celebA.

In [ ]:
# Rutas de descarga y descompresión.
drive_path = '/content/drive/MyDrive/Redes neuronales/Tarea 7. Reconocimiento facial/data/celeba'
zip_name = 'celeba.zip'
zip_in_drive = os.path.join(drive_path, zip_name)
extract_path = '/content/celeba_dataset'

os.makedirs(drive_path, exist_ok=True)

# Descargamos el zip en el drive.
if not os.path.exists(zip_in_drive):
    print("Iniciando descarga...")
    url_celeba = "https://s3-us-west-1.amazonaws.com/udacity-dlnfd/datasets/celeba.zip"

    response = requests.get(url_celeba, stream=True)
    with open(zip_in_drive, "wb") as f:
        for chunk in response.iter_content(chunk_size=1024):
            if chunk: f.write(chunk)
    print("Descarga completada y guardada en Drive.")
else:
    print("El ZIP ya existe en el Drive.")

El ZIP ya existe en el Drive.


In [ ]:
# Descomprimimos en el entorno local.
if not os.path.exists(extract_path):
    print("Descomprimiendo en el entorno local de Colab...")
    try:
        with zipfile.ZipFile(zip_in_drive, 'r') as zip_ref:
            zip_ref.extractall(extract_path)
        print(f"Imágenes descomprimidas en: {extract_path}")
    except zipfile.BadZipFile:
        print("Error")
else:
    print("Imágenes descomprimidas.")

Descomprimiendo en el entorno local de Colab...
Imágenes descomprimidas en: /content/celeba_dataset


In [ ]:
print("Localizando archivos...")
dataset_root = '/content/celeba_dataset'
attr_filename = 'list_attr_celeba.txt'
attr_path = os.path.join(dataset_root, attr_filename)

attr_files = glob.glob(f'{dataset_root}/**/{attr_filename}', recursive=True)

if attr_files:
    attr_path = attr_files[0]
    print(f"Archivo de atributos encontrado en el ZIP: {attr_path}")

else:
    print("No se encontró list_attr_celeba.txt en el disco local.")
    print("Descargando archivo oficial desde Google Drive...")

    # Enlace oficial a list_attr_celeba.txt.
    try:
        !gdown 0B7EVK8r0v71pblRyaVFSWGxPY0U -O {attr_path}
        print(f"Archivo descargado correctamente en: {attr_path}")
    except Exception as e:
        print("Error descargando list_attr_celeba.txt")
        print(e)

# Cargamos el dataframe.
if os.path.exists(attr_path):
    try:
        df_attr = pd.read_csv(attr_path, sep=r'\s+', skiprows=1, index_col=0)
        df_attr.replace(-1, 0, inplace=True)
        print(f"DataFrame cargado exitosamente: {df_attr.shape}")

    except Exception as e:
        print(f"Error leyendo el archivo: {e}")
else:
    print("No se pudo cargar el archivo de atributos. Verifica la descarga.")

# Buscamos imágenes.
img_dirs = glob.glob(f'{dataset_root}/**/img_align_celeba', recursive=True)
if img_dirs:
    img_dir = img_dirs[0]
    print(f"Carpeta de imágenes encontrada: {img_dir}")
else:
    print("NO se encontró la carpeta de imágenes.")

Localizando archivos...
No se encontró list_attr_celeba.txt en el disco local.
Descargando archivo oficial desde Google Drive...
Downloading...
From: https://drive.google.com/uc?id=0B7EVK8r0v71pblRyaVFSWGxPY0U
To: /content/celeba_dataset/list_attr_celeba.txt
100% 26.7M/26.7M [00:00<00:00, 49.6MB/s]
Archivo descargado correctamente en: /content/celeba_dataset/list_attr_celeba.txt
DataFrame cargado exitosamente: (202599, 40)
Carpeta de imágenes encontrada: /content/celeba_dataset/img_align_celeba


## Configuración de prueba rápida.

In [ ]:
if df_attr.index.name != 'filename':
    df_attr.index.name = 'filename'
    df_attr.reset_index(inplace=True)

print(f"Columnas del DataFrame: {df_attr.columns[:5]}")

batch_size = 20
target_size = (224, 224)

# Configuramos generadores de datos.
datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.15)

# Entrenamiento.
train_generator = datagen.flow_from_dataframe(
    dataframe=df_attr,
    directory=img_dir,
    x_col='filename',
    y_col=list(df_attr.columns)[1:],
    target_size=target_size,
    batch_size=batch_size,
    class_mode='raw',
    subset='training')

# Validación.
valid_generator = datagen.flow_from_dataframe(
    dataframe=df_attr,
    directory=img_dir,
    x_col='filename',
    y_col=list(df_attr.columns)[1:],
    target_size=target_size,
    batch_size=batch_size,
    class_mode='raw',
    subset='validation')

Columnas del DataFrame: Index(['filename', '5_o_Clock_Shadow', 'Arched_Eyebrows', 'Attractive',
       'Bags_Under_Eyes'],
      dtype='object')
Found 172210 validated image filenames.
Found 30389 validated image filenames.


In [ ]:
# Arquitectura del modelo.
model.summary(expand_nested=True, line_length=120)

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━
┃ Layer (type)                                        ┃ Output Shape                           ┃               Para
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━
│ conv2d (Conv2D)                                     │ (None, 222, 222, 32)                   │                   
├─────────────────────────────────────────────────────┼────────────────────────────────────────┼───────────────────
│ max_pooling2d (MaxPooling2D)                        │ (None, 111, 111, 32)                   │                   
├─────────────────────────────────────────────────────┼────────────────────────────────────────┼───────────────────
│ conv2d_1 (Conv2D)                                   │ (None, 109, 109, 64)                   │                18,
├─────────────────────────────────────────────────────┼────────────────────────────────────────┼───────────────────
│ max_pooling2d_1 (MaxPooling2D)                      │ (None, 54, 54, 64)                     │                   
├─────────────────────────────────────────────────────┼────────────────────────────────────────┼───────────────────
│ conv2d_2 (Conv2D)                                   │ (None, 52, 52, 128)                    │                73,
├─────────────────────────────────────────────────────┼────────────────────────────────────────┼───────────────────
│ max_pooling2d_2 (MaxPooling2D)                      │ (None, 26, 26, 128)                    │                   
├─────────────────────────────────────────────────────┼────────────────────────────────────────┼───────────────────
│ conv2d_3 (Conv2D)                                   │ (None, 24, 24, 128)                    │               147,
├─────────────────────────────────────────────────────┼────────────────────────────────────────┼───────────────────
│ max_pooling2d_3 (MaxPooling2D)                      │ (None, 12, 12, 128)                    │                   
├─────────────────────────────────────────────────────┼────────────────────────────────────────┼───────────────────
│ flatten (Flatten)                                   │ (None, 18432)                          │                   
├─────────────────────────────────────────────────────┼────────────────────────────────────────┼───────────────────
│ dense (Dense)                                       │ (None, 512)                            │             9,437,
├─────────────────────────────────────────────────────┼────────────────────────────────────────┼───────────────────
│ celeba_output (Dense)                               │ (None, 40)                             │                20,
└─────────────────────────────────────────────────────┴────────────────────────────────────────┴───────────────────

 Total params: 29,097,146 (111.00 MB)

 Trainable params: 9,699,048 (37.00 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 19,398,098 (74.00 MB)

In [ ]:
# Contruimos y entrenamos el modelo.
model = construir_cnn_base(input_shape=(224, 224, 3), num_classes=40)

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['binary_accuracy'])

epochs = 15
history = model.fit(
    train_generator,
    steps_per_epoch=1000,
    validation_data=valid_generator,
    validation_steps=10,
    epochs=epochs)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/15


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


1000/1000 ━━━━━━━━━━━━━━━━━━━━ 47s 41ms/step - binary_accuracy: 0.8377 - loss: 0.3676 - val_binary_accuracy: 0.8895 - val_loss: 0.2550
Epoch 2/15
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 43s 43ms/step - binary_accuracy: 0.8881 - loss: 0.2551 - val_binary_accuracy: 0.8934 - val_loss: 0.2378
Epoch 3/15
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 40s 40ms/step - binary_accuracy: 0.8958 - loss: 0.2364 - val_binary_accuracy: 0.8954 - val_loss: 0.2336
Epoch 4/15
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 41s 41ms/step - binary_accuracy: 0.8995 - loss: 0.2276 - val_binary_accuracy: 0.8985 - val_loss: 0.2279
Epoch 5/15
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 40s 40ms/step - binary_accuracy: 0.9021 - loss: 0.2212 - val_binary_accuracy: 0.8994 - val_loss: 0.2300
Epoch 6/15
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 40s 40ms/step - binary_accuracy: 0.9053 - loss: 0.2148 - val_binary_accuracy: 0.9053 - val_loss: 0.2137
Epoch 7/15
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 40s 40ms/step - binary_accuracy: 0.9074 - loss: 0.2108 - val_binary_accuracy: 0.9078 - val_lo

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:116: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


1000/1000 ━━━━━━━━━━━━━━━━━━━━ 24s 24ms/step - binary_accuracy: 0.9093 - loss: 0.2070 - val_binary_accuracy: 0.9085 - val_loss: 0.2127
Epoch 10/15
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 39s 39ms/step - binary_accuracy: 0.9098 - loss: 0.2044 - val_binary_accuracy: 0.9041 - val_loss: 0.2118
Epoch 11/15
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 40s 40ms/step - binary_accuracy: 0.9109 - loss: 0.2027 - val_binary_accuracy: 0.9062 - val_loss: 0.2136
Epoch 12/15
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 41s 41ms/step - binary_accuracy: 0.9120 - loss: 0.1998 - val_binary_accuracy: 0.9068 - val_loss: 0.2153
Epoch 13/15
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 39s 39ms/step - binary_accuracy: 0.9125 - loss: 0.1997 - val_binary_accuracy: 0.9068 - val_loss: 0.2146
Epoch 14/15
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 40s 40ms/step - binary_accuracy: 0.9143 - loss: 0.1954 - val_binary_accuracy: 0.9090 - val_loss: 0.2092
Epoch 15/15
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 40s 40ms/step - binary_accuracy: 0.9143 - loss: 0.1957 - val_binary_accuracy: 0.9039 - 

In [ ]:
# Guardamos el modelo.
save_path = '/content/drive/MyDrive/Redes neuronales/Tarea 7. Reconocimiento facial/models/celeba_base.h5'
# Creamos carpeta de modelos.
os.makedirs(os.path.dirname(save_path), exist_ok=True)

model.save(save_path)
print(f"\nModelo base guardado en: {save_path}")


Modelo base guardado en: /content/drive/MyDrive/Redes neuronales/Tarea 7. Reconocimiento facial/models/celeba_base.h5
